# 00 · Generador de datos sintéticos — Curso de ML para Salud

> **⚠️ Todos los datos son SINTÉTICOS. No representan ni derivan de pacientes reales.** Se generan con código y una **semilla fija**, así que son siempre los mismos (reproducibles). Su única función es didáctica.

Este cuaderno **crea** el conjunto de datos clínicos que usaremos durante todo el curso. No hace falta descargar nada: aquí nacen los ficheros.

**¿Por qué un cuaderno solo para esto?** Por tres razones:
1. Para que **veas de dónde salen los datos** y entiendas que son inventados de forma controlada.
2. Para que puedas **adaptarlos** (más pacientes, otros patrones) si quieres experimentar.
3. Porque en salud **no se trabaja con datos reales de pacientes sin permisos**; aprender a generar datos sintéticos es, en sí mismo, una buena práctica.

Los demás cuadernos del curso vuelven a generar estos datos en su primera celda, así que cada uno funciona por su cuenta.


## 1. Qué vamos a generar

Vamos a crear seis ficheros CSV (una tabla cada uno). Piensa en un CSV como en una hoja de Excel guardada en texto:

| Fichero | Qué contiene | Para qué lo usaremos |
|---|---|---|
| `pacientes.csv` | 20 000 pacientes con edad, sexo, constantes y analítica, y su **riesgo cardiovascular** | Predecir riesgo (U2–U6) |
| `pacientes_sucio.csv` | La misma tabla pero **con errores a propósito** | Aprender a limpiar datos (U2) |
| `urgencias_diarias.csv` | Ingresos diarios en urgencias durante 2 años | Series temporales (U7) |
| `notas_clinicas.csv` | Notas clínicas cortas con su especialidad | Clasificar texto (U4, U9) |
| `centros.csv` | Catálogo de hospitales y centros de salud | Agrupar centros (U6) |
| `wearable.csv` | Señal de pulsera (pulso, pasos, sueño) | Señal y anomalías (U7, U8) |

No te preocupes por el código: **léelo como si fuera una receta comentada**. Lo importante es entender *qué* datos creamos y *por qué* tienen sentido clínico.


## 2. La receta completa

La siguiente celda define una función, `generar_datos_clinicos()`, que escribe los seis ficheros. Está muy comentada: cada bloque explica qué variable crea y qué relación clínica imita (por ejemplo, que **el tabaquismo y la tensión suben el riesgo**, y que el **HDL lo baja**). Al final, la ejecuta.

> 💡 **Idea clave.** Diseñamos las relaciones **a propósito y de forma realista**. Así, cuando más adelante un modelo "descubra" que fumar sube el riesgo, sabremos que ha aprendido algo verdadero, y podremos comprobar si acierta.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## 3. Conozcamos a nuestros pacientes

Ya tenemos los ficheros. Vamos a **abrir** la tabla principal y mirarla, que es lo primero que se hace siempre con unos datos nuevos. Cargamos `pacientes.csv` en un *DataFrame* (una tabla de pandas) y mostramos las primeras filas.


In [ ]:
import pandas as pd

pacientes = pd.read_csv("pacientes.csv")
print("Filas y columnas:", pacientes.shape)   # (nº de pacientes, nº de variables)
pacientes.head()

Cada **fila** es un paciente; cada **columna**, una característica. Fíjate en las dos últimas columnas, que son las que querremos **predecir** más adelante:

- `riesgo_cv_10a`: el riesgo cardiovascular a 10 años, en **porcentaje** (un número continuo → problema de *regresión*).
- `evento_cv`: si el paciente sufre o no un evento cardiovascular, **0 o 1** (una categoría → problema de *clasificación*).

Veamos ahora un **resumen numérico** de cada columna: medias, mínimos, máximos… Nos dice de un vistazo si los valores son plausibles (por ejemplo, que la edad esté entre 18 y 95).


In [ ]:
pacientes.describe(include="all").round(1)

**Cómo leer esta tabla:** para cada columna numérica, `mean` es la media, `std` la dispersión, y `min`/`max` los extremos. Comprobamos que la edad, la tensión o la glucemia caen en rangos clínicos razonables. Para las columnas de texto (sexo, tabaquismo) aparece cuántos valores distintos hay (`unique`) y el más frecuente (`top`).


### 3.1 ¿Cómo de frecuente es el evento que queremos predecir?

En medicina, saber la **prevalencia** (qué proporción de pacientes tiene el evento) es fundamental: condiciona todo lo demás. La calculamos.


In [ ]:
prevalencia = pacientes["evento_cv"].mean()
print(f"Prevalencia de evento cardiovascular: {prevalencia:.1%}")
print(f"Pacientes con diabetes: {pacientes['diabetes'].mean():.1%}")

Alrededor de un **19 %** de la cohorte sufre el evento. Es un problema **desbalanceado** (muchos más "sanos" que "casos"), algo muy habitual en clínica que tendremos muy en cuenta al **evaluar** los modelos (U3): en estos casos, la exactitud a secas engaña.


### 3.2 Una imagen vale más: ¿sube el riesgo con la edad?

Vamos a **dibujar** la relación entre la edad y el riesgo cardiovascular. Esperamos ver que el riesgo **crece** con la edad (es lo que pusimos en la receta y lo que ocurre en la realidad). Usamos un diagrama de dispersión (cada punto es un paciente).


In [ ]:
import matplotlib.pyplot as plt

muestra = pacientes.sample(2000, random_state=1)   # 2000 puntos bastan para ver el patrón
plt.figure(figsize=(7, 4))
plt.scatter(muestra["edad"], muestra["riesgo_cv_10a"], s=8, alpha=0.4)
plt.xlabel("Edad (años)")
plt.ylabel("Riesgo cardiovascular a 10 años (%)")
plt.title("Relación entre edad y riesgo cardiovascular (datos sintéticos)")
plt.show()

**Lo que vemos:** una nube que **sube de izquierda a derecha**. A mayor edad, mayor riesgo, con bastante variabilidad entre personas de la misma edad (porque el riesgo también depende del tabaco, la tensión, etc.). Justo lo que esperábamos: los datos se comportan de forma clínicamente creíble.


### 3.3 ¿Fumar cambia el riesgo?

Comparamos la proporción de eventos según el hábito tabáquico. Agrupamos por `tabaquismo` y calculamos la media de `evento_cv` en cada grupo.


In [ ]:
(pacientes.groupby("tabaquismo")["evento_cv"].mean().mul(100).round(1)
 .rename("% con evento CV"))

**Lo que vemos:** el porcentaje de eventos es más alto en fumadores **activos** que en **ex**-fumadores, y mínimo en quienes **nunca** fumaron. Un gradiente claro y en la dirección correcta. Este tipo de comprobaciones sencillas —mirar si los datos "tienen sentido"— es el corazón del **análisis exploratorio** que haremos en U2.


## 4. Un vistazo rápido a los demás ficheros

No hace falta analizarlos a fondo aquí (cada uno tiene su unidad), pero conviene **abrirlos una vez** para saber qué pinta tienen.


In [ ]:
urgencias = pd.read_csv("urgencias_diarias.csv")
print("Urgencias — filas:", len(urgencias))
urgencias.head()

In [ ]:
notas = pd.read_csv("notas_clinicas.csv")
print("Notas clínicas — filas:", len(notas))
notas.head()

In [ ]:
centros = pd.read_csv("centros.csv")
print("Centros — filas:", len(centros))
centros.head()

## 5. Qué hemos aprendido

- Los datos del curso son **sintéticos**, creados con una receta **comentada y reproducible**.
- La tabla central, `pacientes.csv`, tiene 20 000 pacientes y dos objetivos a predecir: `riesgo_cv_10a` (regresión) y `evento_cv` (clasificación, prevalencia ≈ 19 %).
- Antes de modelar nada, **miramos los datos**: forma, resumen numérico y un par de gráficas para confirmar que se comportan de forma clínicamente razonable.

> 🤖 **Con un asistente de IA** podrías pedir todo esto en lenguaje natural, por ejemplo: *"Carga pacientes.csv, dime cuántas filas tiene, resume cada columna y dibuja el riesgo frente a la edad"*. El asistente escribe el código; tú aportas la **pregunta clínica** y la **interpretación**. Eso es exactamente lo que practicaremos en la U10.

**Siguiente paso:** en la **U2** usaremos `pacientes_sucio.csv` para aprender a **detectar y limpiar errores** en los datos, que es donde se va la mitad del trabajo real.
